# Representation Learning: PCA and Autoencoders
### CIFAR-10 (Grayscale) and MNIST Datasets

**Assignment Overview:**
- **Task 1:** Standard PCA vs Randomized PCA with Logistic Regression + ROC curves + SNR
- **Task 2:** Tied-weight single-layer linear autoencoder — comparison with PCA eigenvectors
- **Task 3:** Deep convolutional autoencoder vs shallow autoencoders — SNR comparison

---

## Setup: Imports

In [5]:
!pip install --upgrade pip
!pip install tensorflow

  ERROR: HTTP error 403 while getting https://files.pythonhosted.org/packages/3a/eb/fea4d1d51c49832120f7f285d07306db3960f423a2612c6057caf3e8196f/pip-26.1.1-py3-none-any.whl.metadata

[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: 403 Client Error: Forbidden for url: https://files.pythonhosted.org/packages/3a/eb/fea4d1d51c49832120f7f285d07306db3960f423a2612c6057caf3e8196f/pip-26.1.1-py3-none-any.whl.metadata
  ERROR: HTTP error 403 while getting https://files.pythonhosted.org/packages/ec/76/b649b02a243c09f0348199dbd81408c1558cfbec2b6d77d820c428a95254/tensorflow-2.21.0-cp311-cp311-macosx_12_0_arm64.whl.metadata

[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: 403 Client Error: Forbidden for url: https://files.pythonhosted.org/packages/ec/76/b649b02a243c09f0348199dbd81408c1558cfbec2b6d77d820c428a95254/tensorflow-2.21.0-cp311-cp311-macosx_12_0_arm64.whl

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

# Sklearn
from sklearn.decomposition import PCA
from sklearn.decomposition import TruncatedSVD          # Randomized PCA via TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_curve, auc,
                              accuracy_score,
                              classification_report)
from sklearn.preprocessing import label_binarize

# Image processing
from skimage.transform import resize

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

ModuleNotFoundError: No module named 'tensorflow'

---
## Data Loading and Preprocessing

**Steps:**
1. Load CIFAR-10 (convert to grayscale) and MNIST
2. Resize all images to **28×28**
3. Normalize intensity values to range **[50, 200]**
4. Split: **70% train | 20% validation | 10% test** (from the combined full dataset)

In [ ]:
# ─── Helper functions ────────────────────────────────────────────────────────

def rgb_to_gray(images):
    """Convert RGB images (N,H,W,3) to grayscale (N,H,W) using luminance formula."""
    return (0.2989 * images[..., 0] +
            0.5870 * images[..., 1] +
            0.1140 * images[..., 2]).astype(np.float32)

def resize_images(images, target_size=(28, 28)):
    """Resize a batch of grayscale images to target_size."""
    resized = np.zeros((len(images), *target_size), dtype=np.float32)
    for i, img in enumerate(images):
        resized[i] = resize(img, target_size,
                            anti_aliasing=True,
                            preserve_range=True)
    return resized

def normalize_intensity(images, low=50, high=200):
    """
    Linearly scale pixel values so that the global min maps to `low`
    and the global max maps to `high`.
    """
    x_min = images.min()
    x_max = images.max()
    return low + (images - x_min) / (x_max - x_min + 1e-8) * (high - low)

def split_dataset(X, y, train_frac=0.70, val_frac=0.20, seed=SEED):
    """Shuffle and split into train / val / test (test = remainder)."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(X))
    n_train = int(len(X) * train_frac)
    n_val   = int(len(X) * val_frac)
    tr_idx  = idx[:n_train]
    va_idx  = idx[n_train:n_train + n_val]
    te_idx  = idx[n_train + n_val:]
    return (X[tr_idx], y[tr_idx],
            X[va_idx], y[va_idx],
            X[te_idx], y[te_idx])

def preprocess_dataset(X_raw, y_raw, name, is_rgb=False):
    """Full preprocessing pipeline: gray → resize → normalize → split."""
    print(f'\n[{name}] Raw shape: {X_raw.shape}, dtype: {X_raw.dtype}')
    X = X_raw.astype(np.float32)
    if is_rgb:
        X = rgb_to_gray(X)                 # (N,32,32)
    if X.shape[1] != 28 or X.shape[2] != 28:
        X = resize_images(X, (28, 28))     # (N,28,28)
    X = normalize_intensity(X)             # values in [50, 200]
    print(f'[{name}] After preprocessing — shape: {X.shape}, '
          f'min: {X.min():.1f}, max: {X.max():.1f}')
    Xtr, ytr, Xva, yva, Xte, yte = split_dataset(X, y_raw)
    print(f'[{name}] Train: {Xtr.shape[0]} | Val: {Xva.shape[0]} | Test: {Xte.shape[0]}')
    return Xtr, ytr, Xva, yva, Xte, yte

In [ ]:
# ─── Load datasets ────────────────────────────────────────────────────────────

# MNIST (already grayscale 28×28)
(mnist_X_train_raw, mnist_y_train_raw), (mnist_X_test_raw, mnist_y_test_raw) = \
    keras.datasets.mnist.load_data()
mnist_X_all = np.concatenate([mnist_X_train_raw, mnist_X_test_raw], axis=0)
mnist_y_all = np.concatenate([mnist_y_train_raw, mnist_y_test_raw], axis=0)

# CIFAR-10 (RGB 32×32 → convert to gray)
(cifar_X_train_raw, cifar_y_train_raw), (cifar_X_test_raw, cifar_y_test_raw) = \
    keras.datasets.cifar10.load_data()
cifar_X_all = np.concatenate([cifar_X_train_raw, cifar_X_test_raw], axis=0)
cifar_y_all = np.concatenate(
    [cifar_y_train_raw.ravel(), cifar_y_test_raw.ravel()], axis=0)

# ─── Preprocess ───────────────────────────────────────────────────────────────
(
 mnist_Xtr, mnist_ytr,
 mnist_Xva, mnist_yva,
 mnist_Xte, mnist_yte
) = preprocess_dataset(mnist_X_all, mnist_y_all, 'MNIST', is_rgb=False)

(
 cifar_Xtr, cifar_ytr,
 cifar_Xva, cifar_yva,
 cifar_Xte, cifar_yte
) = preprocess_dataset(cifar_X_all, cifar_y_all, 'CIFAR-10', is_rgb=True)

# Flat representations for PCA / classifiers
IMG_DIM = 28 * 28   # 784

mnist_Xtr_flat = mnist_Xtr.reshape(-1, IMG_DIM)
mnist_Xva_flat = mnist_Xva.reshape(-1, IMG_DIM)
mnist_Xte_flat = mnist_Xte.reshape(-1, IMG_DIM)

cifar_Xtr_flat = cifar_Xtr.reshape(-1, IMG_DIM)
cifar_Xva_flat = cifar_Xva.reshape(-1, IMG_DIM)
cifar_Xte_flat = cifar_Xte.reshape(-1, IMG_DIM)

In [ ]:
# ─── Visualise a few samples from each dataset ────────────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
fig.suptitle('Sample Images After Preprocessing (intensity ∈ [50, 200])', fontsize=13)
for i in range(10):
    axes[0, i].imshow(mnist_Xtr[i], cmap='gray', vmin=50, vmax=200)
    axes[0, i].axis('off')
    axes[0, i].set_title(f'MNIST\n{mnist_ytr[i]}', fontsize=8)
    axes[1, i].imshow(cifar_Xtr[i], cmap='gray', vmin=50, vmax=200)
    axes[1, i].axis('off')
    axes[1, i].set_title(f'CIFAR\n{cifar_ytr[i]}', fontsize=8)
plt.tight_layout()
plt.show()

---
## Task 1: Standard PCA vs Randomized PCA

**Procedure:**
1. Fit PCA on **70% of the training set** (i.e. a subset of the 70% training split)
2. Extract top-30 principal components (eigenvectors)
3. Project train / test sets to the 30-D space
4. Train `LogisticRegression` and evaluate on the test set
5. Plot multi-class ROC curves (one-vs-rest)
6. Reconstruct test images and compute average SNR in dB
7. Repeat with `TruncatedSVD` (Randomized PCA) and compare

In [ ]:
# ─── Helper: SNR computation ──────────────────────────────────────────────────
def compute_snr_db(original, reconstructed):
    """
    Average SNR (dB) over a batch of images.
    SNR_i = 10 * log10( ||x_i||^2 / ||x_i - x_hat_i||^2 )
    Returns the mean SNR across the batch.
    """
    original      = original.reshape(len(original), -1).astype(np.float64)
    reconstructed = reconstructed.reshape(len(reconstructed), -1).astype(np.float64)
    signal_power  = np.sum(original ** 2, axis=1)
    noise_power   = np.sum((original - reconstructed) ** 2, axis=1)
    # Avoid division by zero
    snr_db = np.where(noise_power > 1e-12,
                      10 * np.log10(signal_power / (noise_power + 1e-12)),
                      np.inf)
    return np.nanmean(snr_db)

# ─── Helper: multi-class ROC (one-vs-rest) ────────────────────────────────────
def plot_roc_curves(y_true, y_score, n_classes=10, title='ROC Curves', ax=None):
    """
    Plot per-class ROC + macro-average ROC.
    y_score: (N, 10) probability matrix from predict_proba.
    """
    y_bin = label_binarize(y_true, classes=list(range(n_classes)))
    fpr_all, tpr_all, roc_auc_all = {}, {}, {}
    for c in range(n_classes):
        fpr_all[c], tpr_all[c], _ = roc_curve(y_bin[:, c], y_score[:, c])
        roc_auc_all[c] = auc(fpr_all[c], tpr_all[c])
    # Macro average
    all_fpr  = np.unique(np.concatenate([fpr_all[c] for c in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for c in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr_all[c], tpr_all[c])
    mean_tpr /= n_classes
    macro_auc = auc(all_fpr, mean_tpr)

    if ax is None:
        _, ax = plt.subplots(figsize=(7, 5))
    cmap = plt.cm.get_cmap('tab10', n_classes)
    for c in range(n_classes):
        ax.plot(fpr_all[c], tpr_all[c], color=cmap(c), lw=1,
                label=f'Class {c} (AUC={roc_auc_all[c]:.2f})')
    ax.plot(all_fpr, mean_tpr, 'k--', lw=2,
            label=f'Macro avg (AUC={macro_auc:.2f})')
    ax.plot([0, 1], [0, 1], 'gray', linestyle=':', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(title)
    ax.legend(loc='lower right', fontsize=7, ncol=2)
    return macro_auc

In [ ]:
# ─── Task 1: PCA pipeline ─────────────────────────────────────────────────────
N_COMPONENTS = 30

def run_pca_pipeline(Xtr, ytr, Xte, yte, name,
                     pca_subset_frac=0.70, use_randomized=False):
    """
    Full PCA pipeline:
    1. Fit PCA on pca_subset_frac of training data
    2. Project train + test
    3. Fit logistic regression on projected train
    4. Evaluate on projected test — accuracy + ROC
    5. Reconstruct test images and compute SNR
    Returns dict with results + the fitted pca object.
    """
    # --- 1. Subset for PCA fitting ---
    n_pca = int(len(Xtr) * pca_subset_frac)
    rng   = np.random.default_rng(SEED)
    idx   = rng.choice(len(Xtr), n_pca, replace=False)
    Xtr_pca_fit = Xtr[idx]

    # --- 2. Center (mean subtract) before PCA ---
    mean_vec = Xtr_pca_fit.mean(axis=0)
    Xtr_c    = Xtr - mean_vec          # centered full training set
    Xte_c    = Xte - mean_vec
    Xfit_c   = Xtr_pca_fit - mean_vec  # centered PCA fitting subset

    if use_randomized:
        pca = TruncatedSVD(n_components=N_COMPONENTS, random_state=SEED)
        pca_label = 'Randomized PCA (TruncatedSVD)'
    else:
        pca = PCA(n_components=N_COMPONENTS, random_state=SEED)
        pca_label = 'Standard PCA'

    pca.fit(Xfit_c)

    # --- 3. Project ---
    Xtr_proj = pca.transform(Xtr_c)
    Xte_proj = pca.transform(Xte_c)

    # --- 4. Logistic Regression ---
    clf = LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=-1)
    clf.fit(Xtr_proj, ytr)
    y_pred  = clf.predict(Xte_proj)
    y_score = clf.predict_proba(Xte_proj)
    acc = accuracy_score(yte, y_pred)
    print(f'\n[{name}] {pca_label} — Test Accuracy: {acc*100:.2f}%')

    # --- 5. Reconstruct for SNR ---
    if use_randomized:
        Xte_recon = pca.inverse_transform(Xte_proj) + mean_vec
    else:
        Xte_recon = pca.inverse_transform(Xte_proj) + mean_vec
    snr = compute_snr_db(Xte + mean_vec, Xte_recon)  # compare against original
    # Oops — Xte is already centered (Xte_c = Xte - mean), Xte original = Xte_flat
    # Let's recalculate properly:
    Xte_original = Xte + mean_vec  # undo centering
    snr = compute_snr_db(Xte_original, Xte_recon)
    print(f'[{name}] {pca_label} — Average Test SNR: {snr:.2f} dB')

    return dict(pca=pca, clf=clf, acc=acc, snr=snr,
                y_pred=y_pred, y_score=y_score,
                Xte_recon=Xte_recon, mean_vec=mean_vec,
                label=pca_label, name=name)


# Run for both datasets
print('=' * 60)
print('MNIST')
print('=' * 60)
mnist_pca_res   = run_pca_pipeline(mnist_Xtr_flat, mnist_ytr,
                                    mnist_Xte_flat, mnist_yte, 'MNIST')
mnist_rpca_res  = run_pca_pipeline(mnist_Xtr_flat, mnist_ytr,
                                    mnist_Xte_flat, mnist_yte, 'MNIST',
                                    use_randomized=True)
print('\n' + '=' * 60)
print('CIFAR-10')
print('=' * 60)
cifar_pca_res  = run_pca_pipeline(cifar_Xtr_flat, cifar_ytr,
                                   cifar_Xte_flat, cifar_yte, 'CIFAR-10')
cifar_rpca_res = run_pca_pipeline(cifar_Xtr_flat, cifar_ytr,
                                   cifar_Xte_flat, cifar_yte, 'CIFAR-10',
                                   use_randomized=True)

In [ ]:
# ─── Task 1: ROC Curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Task 1 — ROC Curves (One-vs-Rest, 10 Classes)', fontsize=14)

plot_roc_curves(mnist_yte,  mnist_pca_res['y_score'],
                title=f'MNIST — Standard PCA (Acc={mnist_pca_res["acc"]*100:.1f}%)',
                ax=axes[0, 0])
plot_roc_curves(mnist_yte,  mnist_rpca_res['y_score'],
                title=f'MNIST — Randomized PCA (Acc={mnist_rpca_res["acc"]*100:.1f}%)',
                ax=axes[0, 1])
plot_roc_curves(cifar_yte,  cifar_pca_res['y_score'],
                title=f'CIFAR-10 — Standard PCA (Acc={cifar_pca_res["acc"]*100:.1f}%)',
                ax=axes[1, 0])
plot_roc_curves(cifar_yte,  cifar_rpca_res['y_score'],
                title=f'CIFAR-10 — Randomized PCA (Acc={cifar_rpca_res["acc"]*100:.1f}%)',
                ax=axes[1, 1])
plt.tight_layout()
plt.savefig('task1_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Task 1: Visualise top-30 eigenvectors (Principal Components) ─────────────
def plot_eigenvectors(pca, title, n_components=30):
    if hasattr(pca, 'components_'):
        components = pca.components_   # shape (n_components, 784)
    else:
        components = pca.components_

    fig, axes = plt.subplots(3, 10, figsize=(15, 5))
    fig.suptitle(title, fontsize=13)
    for i, ax in enumerate(axes.flat):
        if i < n_components:
            ax.imshow(components[i].reshape(28, 28), cmap='gray')
            ax.set_title(f'PC {i+1}', fontsize=7)
        ax.axis('off')
    plt.tight_layout()
    return fig

fig1 = plot_eigenvectors(mnist_pca_res['pca'],
                          'MNIST — Top-30 Standard PCA Eigenvectors')
plt.savefig('mnist_pca_eigenvectors.png', dpi=150, bbox_inches='tight')
plt.show()

fig2 = plot_eigenvectors(cifar_pca_res['pca'],
                          'CIFAR-10 — Top-30 Standard PCA Eigenvectors')
plt.savefig('cifar_pca_eigenvectors.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Task 1: Reconstructed test images + SNR summary ─────────────────────────
def show_reconstructions(Xte_orig_flat, recon_flat, mean_vec,
                          n_show=8, title='', snr_db=None):
    """Display original vs reconstructed images side by side."""
    fig, axes = plt.subplots(2, n_show, figsize=(n_show * 1.5, 4))
    sup = f'{title}' + (f'  |  Avg SNR = {snr_db:.2f} dB' if snr_db else '')
    fig.suptitle(sup, fontsize=11)
    orig_full  = Xte_orig_flat + mean_vec   # restore mean
    for i in range(n_show):
        axes[0, i].imshow(orig_full[i].reshape(28, 28), cmap='gray')
        axes[0, i].axis('off')
        axes[0, i].set_title('Orig', fontsize=7)
        axes[1, i].imshow(recon_flat[i].reshape(28, 28), cmap='gray')
        axes[1, i].axis('off')
        axes[1, i].set_title('Recon', fontsize=7)
    plt.tight_layout()
    plt.show()

show_reconstructions(mnist_Xte_flat - mnist_pca_res['mean_vec'],
                     mnist_pca_res['Xte_recon'],
                     mnist_pca_res['mean_vec'],
                     title='MNIST — Standard PCA Reconstruction',
                     snr_db=mnist_pca_res['snr'])

show_reconstructions(cifar_Xte_flat - cifar_pca_res['mean_vec'],
                     cifar_pca_res['Xte_recon'],
                     cifar_pca_res['mean_vec'],
                     title='CIFAR-10 — Standard PCA Reconstruction',
                     snr_db=cifar_pca_res['snr'])

# ─── Task 1: Summary table ────────────────────────────────────────────────────
print('\n' + '=' * 70)
print(f'{"Dataset":<12} {"Method":<25} {"Test Accuracy":>14} {"Avg SNR (dB)":>14}')
print('-' * 70)
for res in [mnist_pca_res, mnist_rpca_res, cifar_pca_res, cifar_rpca_res]:
    print(f'{res["name"]:<12} {res["label"]:<25} '
          f'{res["acc"]*100:>13.2f}% {res["snr"]:>14.2f}')
print('=' * 70)

### Task 1 — Discussion

| Observation | Explanation |
|---|---|
| **Standard vs Randomized PCA** | Both should yield nearly identical results. Randomized PCA (TruncatedSVD) uses a stochastic algorithm and is much faster on large datasets. |
| **MNIST vs CIFAR-10 accuracy** | MNIST is simpler (structured digits) → higher accuracy. CIFAR-10 grayscale loses colour information → harder classification. |
| **SNR** | More components → higher SNR. With only 30 components the reconstruction is lossy but captures the dominant structure. |

---
## Task 2: Tied-Weight Linear Autoencoder (30 Nodes)

**Design:**
- Input: mean-and-variance-normalized flattened images (784-D)
- Encoder: $z = W^T x + b_e$  (linear)
- Decoder: $\hat{x} = W z + b_d$  (tied weights, linear)
- **Constraint:** each column of $W$ has unit $\ell_2$ norm (enforced in the gradient update)

**Theory:** The optimal weight matrix for a tied linear autoencoder minimizing MSE is the matrix whose columns span the same subspace as the top PCA eigenvectors. The individual weight vectors may differ by rotation within that subspace.

**Comparison:** We quantify the alignment between PCA eigenvectors and AE weight vectors using the **subspace angle** and **average cosine similarity**.

In [ ]:
# ─── Tied-weight autoencoder model ───────────────────────────────────────────

class UnitNormConstraint(tf.keras.constraints.Constraint):
    """Projects each column of the weight matrix to unit L2 norm."""
    def __call__(self, w):
        # w shape: (input_dim, latent_dim)
        return tf.math.l2_normalize(w, axis=0)


class TiedLinearAutoencoder(tf.keras.Model):
    """
    Single-layer linear autoencoder with tied, unit-norm weights.
    Architecture:
        Encoder: z = W^T x + b_enc   (W: input_dim × latent_dim)
        Decoder: x_hat = W z + b_dec  (same W, transposed)
    """
    def __init__(self, input_dim, latent_dim=30, **kwargs):
        super().__init__(**kwargs)
        self.latent_dim = latent_dim
        self.input_dim  = input_dim
        self.W = self.add_weight(
            name='W',
            shape=(input_dim, latent_dim),
            initializer='glorot_uniform',
            constraint=UnitNormConstraint(),
            trainable=True
        )
        self.b_enc = self.add_weight(
            name='b_enc', shape=(latent_dim,),
            initializer='zeros', trainable=True
        )
        self.b_dec = self.add_weight(
            name='b_dec', shape=(input_dim,),
            initializer='zeros', trainable=True
        )

    def encode(self, x):
        """Encode: z = x W + b_enc  (x: N×D, W: D×K → z: N×K)"""
        return tf.matmul(x, self.W) + self.b_enc

    def decode(self, z):
        """Decode: x_hat = z W^T + b_dec"""
        return tf.matmul(z, tf.transpose(self.W)) + self.b_dec

    def call(self, x, training=False):
        z     = self.encode(x)
        x_hat = self.decode(z)
        return x_hat

    def get_weight_matrix(self):
        """Return the unit-norm-constrained W (D × K)."""
        return tf.math.l2_normalize(self.W, axis=0).numpy()


def build_and_train_tied_ae(Xtr_flat, Xva_flat, name, input_dim=IMG_DIM,
                             latent_dim=30, epochs=60, batch_size=256):
    """
    Standardise input, then train the tied linear AE.
    Returns the trained model + scaler parameters.
    """
    # Mean & variance normalization
    mu  = Xtr_flat.mean(axis=0, keepdims=True)
    sig = Xtr_flat.std(axis=0, keepdims=True) + 1e-8
    Xtr_n = (Xtr_flat - mu) / sig
    Xva_n = (Xva_flat - mu) / sig

    model = TiedLinearAutoencoder(input_dim, latent_dim)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')

    cb = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
    history = model.fit(
        Xtr_n, Xtr_n,
        validation_data=(Xva_n, Xva_n),
        epochs=epochs, batch_size=batch_size,
        callbacks=[cb], verbose=0
    )
    val_loss = min(history.history['val_loss'])
    print(f'[{name}] Tied AE training done | Best val MSE: {val_loss:.4f}')
    return model, mu, sig


print('Training tied linear autoencoders...')
mnist_ae2, mnist_mu2, mnist_sig2 = build_and_train_tied_ae(
    mnist_Xtr_flat, mnist_Xva_flat, 'MNIST')
cifar_ae2, cifar_mu2, cifar_sig2 = build_and_train_tied_ae(
    cifar_Xtr_flat, cifar_Xva_flat, 'CIFAR-10')
print('Done.')

In [ ]:
# ─── Task 2: Compare PCA eigenvectors vs AE weight vectors ───────────────────

def compare_pca_ae(pca_result, ae_model, dataset_name, n_comp=30):
    """
    1. Get PCA eigenvectors (components_) and AE weight columns
    2. Visualise both as 28×28 grayscale patches
    3. Compute average |cosine similarity| between the two bases
    4. Compute principal angles between the subspaces
    """
    # PCA eigenvectors: rows of components_ (each is a 784-D vector)
    pca_vectors = pca_result['pca'].components_          # (30, 784)
    # AE weight vectors: columns of W (each column = one encoder direction)
    ae_W = ae_model.get_weight_matrix()                  # (784, 30)
    ae_vectors = ae_W.T                                  # (30, 784)

    # ── Cosine similarity matrix ──────────────────────────────────────────────
    # Normalise rows
    pca_n = pca_vectors / (np.linalg.norm(pca_vectors, axis=1, keepdims=True) + 1e-12)
    ae_n  = ae_vectors  / (np.linalg.norm(ae_vectors,  axis=1, keepdims=True) + 1e-12)
    cos_mat = np.abs(pca_n @ ae_n.T)   # (30, 30)
    avg_cos = cos_mat.max(axis=1).mean()   # best match per PCA vector

    # ── Principal angles between subspaces ───────────────────────────────────
    # Use SVD of (Q_pca^T Q_ae) where Q are orthonormal bases
    Q_pca, _ = np.linalg.qr(pca_vectors.T)   # (784, 30)
    Q_ae, _  = np.linalg.qr(ae_W)             # (784, 30)
    sv = np.linalg.svd(Q_pca.T @ Q_ae, compute_uv=False)
    sv = np.clip(sv, -1, 1)
    principal_angles_deg = np.degrees(np.arccos(sv))

    print(f'\n[{dataset_name}] PCA ↔ AE Comparison')
    print(f'  Avg max |cosine similarity| (best-match): {avg_cos:.4f}')
    print(f'  Principal angles (deg): min={principal_angles_deg.min():.2f}, '
          f'max={principal_angles_deg.max():.2f}, '
          f'mean={principal_angles_deg.mean():.2f}')

    # ── Side-by-side visualisation ────────────────────────────────────────────
    fig, axes = plt.subplots(4, 10, figsize=(16, 7))
    fig.suptitle(
        f'{dataset_name} — PCA Eigenvectors (top 2 rows) vs '
        f'AE Weight Vectors (bottom 2 rows)\n'
        f'Avg max |cosine sim| = {avg_cos:.4f}',
        fontsize=11)
    for i in range(n_comp):
        row_pca = i // 10
        col     = i %  10
        axes[row_pca, col].imshow(
            pca_vectors[i].reshape(28, 28), cmap='RdBu')
        axes[row_pca, col].set_title(f'PC{i+1}', fontsize=6)
        axes[row_pca, col].axis('off')

        row_ae = 2 + i // 10
        axes[row_ae, col].imshow(
            ae_vectors[i].reshape(28, 28), cmap='RdBu')
        axes[row_ae, col].set_title(f'AE{i+1}', fontsize=6)
        axes[row_ae, col].axis('off')

    plt.tight_layout()
    plt.savefig(f'{dataset_name.lower().replace("-","_")}_task2_comparison.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    return avg_cos, principal_angles_deg


mnist_cos2, mnist_pa2 = compare_pca_ae(mnist_pca_res, mnist_ae2, 'MNIST')
cifar_cos2, cifar_pa2 = compare_pca_ae(cifar_pca_res, cifar_ae2, 'CIFAR-10')

In [ ]:
# ─── Task 2: Logistic Regression on AE features ───────────────────────────────

def ae2_classify(ae_model, mu, sig,
                 Xtr_flat, ytr, Xte_flat, yte, name):
    # Standardise
    Xtr_n = (Xtr_flat - mu) / sig
    Xte_n = (Xte_flat - mu) / sig

    # Encode
    Ztr = ae_model.encode(Xtr_n.astype(np.float32)).numpy()
    Zte = ae_model.encode(Xte_n.astype(np.float32)).numpy()

    # Classify
    clf = LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=-1)
    clf.fit(Ztr, ytr)
    y_pred  = clf.predict(Zte)
    y_score = clf.predict_proba(Zte)
    acc = accuracy_score(yte, y_pred)
    print(f'[{name}] Tied AE (Task 2) Logistic Reg — Test Accuracy: {acc*100:.2f}%')

    # Reconstruction SNR
    Xte_n_tf  = tf.constant(Xte_n.astype(np.float32))
    Xte_recon_n = ae_model(Xte_n_tf, training=False).numpy()
    # Undo standardisation
    Xte_recon = Xte_recon_n * sig + mu
    snr = compute_snr_db(Xte_flat, Xte_recon)
    print(f'[{name}] Tied AE (Task 2) Average Test SNR: {snr:.2f} dB')
    return acc, snr, y_pred, y_score


mnist_ae2_acc, mnist_ae2_snr, mnist_ae2_pred, mnist_ae2_score = ae2_classify(
    mnist_ae2, mnist_mu2, mnist_sig2,
    mnist_Xtr_flat, mnist_ytr, mnist_Xte_flat, mnist_yte, 'MNIST')

cifar_ae2_acc, cifar_ae2_snr, cifar_ae2_pred, cifar_ae2_score = ae2_classify(
    cifar_ae2, cifar_mu2, cifar_sig2,
    cifar_Xtr_flat, cifar_ytr, cifar_Xte_flat, cifar_yte, 'CIFAR-10')

In [ ]:
# ─── Task 2: ROC curves for AE classifier ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Task 2 — ROC Curves: Tied Linear AE Features', fontsize=13)
plot_roc_curves(mnist_yte, mnist_ae2_score,
                title=f'MNIST — Tied AE (Acc={mnist_ae2_acc*100:.1f}%)',
                ax=axes[0])
plot_roc_curves(cifar_yte, cifar_ae2_score,
                title=f'CIFAR-10 — Tied AE (Acc={cifar_ae2_acc*100:.1f}%)',
                ax=axes[1])
plt.tight_layout()
plt.savefig('task2_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── Task 2: Comparison summary ───────────────────────────────────────────────
print('\n' + '=' * 75)
print(f'{"Dataset":<12} {"Method":<30} {"Test Acc":>10} {"SNR (dB)":>10}')
print('-' * 75)
rows = [
    ('MNIST',    'Standard PCA + LR',   mnist_pca_res['acc'],  mnist_pca_res['snr']),
    ('MNIST',    'Rand PCA + LR',       mnist_rpca_res['acc'], mnist_rpca_res['snr']),
    ('MNIST',    'Tied Linear AE + LR', mnist_ae2_acc,         mnist_ae2_snr),
    ('CIFAR-10', 'Standard PCA + LR',   cifar_pca_res['acc'],  cifar_pca_res['snr']),
    ('CIFAR-10', 'Rand PCA + LR',       cifar_rpca_res['acc'], cifar_rpca_res['snr']),
    ('CIFAR-10', 'Tied Linear AE + LR', cifar_ae2_acc,         cifar_ae2_snr),
]
for ds, method, acc, snr in rows:
    print(f'{ds:<12} {method:<30} {acc*100:>9.2f}% {snr:>10.2f}')
print('=' * 75)

### Task 2 — Discussion

**Relationship between PCA eigenvectors and tied linear AE weights:**

> A tied linear autoencoder with MSE loss is theoretically equivalent to PCA: the optimal encoder weight matrix spans the **same subspace** as the matrix of top-$K$ PCA eigenvectors. However, the individual weight vectors may differ by an **orthogonal rotation** within that $K$-dimensional subspace — hence the average cosine similarity between matched vectors is typically high but the vectors are not identical.

| Metric | Interpretation |
|---|---|
| **Avg max cosine similarity** | Close to 1.0 → each PCA eigenvector is nearly aligned with the closest AE vector. Values < 1 reflect the rotational ambiguity. |
| **Subspace (principal) angles** | Small angles → the two 30-D subspaces are nearly identical, confirming the theoretical equivalence. |

In practice, with finite data and SGD optimisation, small numerical differences appear, but the overall subspace alignment is very strong.

---
## Task 3: Deep Architectures — SNR Comparison

Three architectures compared:

| Model | Encoder | Decoder | Latent Dim |
|---|---|---|---|
| **(A) Deep Conv AE** | Conv2D stack | Conv2DTranspose stack | 30 |
| **(B) Single hidden layer AE** | Dense(30, sigmoid) | Dense(784, linear) | 30 |
| **(C) 3-layer AE** | Dense(10,σ)→Dense(10,σ)→Dense(10,σ) | Dense(10,σ)→Dense(10,σ)→Dense(784,linear) | 10 (last enc layer) |

In [ ]:
# ─── Common preprocessing for Task 3 ─────────────────────────────────────────
# Normalise to [0,1] for neural network training, then rescale SNR back to [50,200]

def normalize_01(X, low=50.0, high=200.0):
    """Map [50,200] → [0,1]."""
    return (X - low) / (high - low)

def denormalize_01(X, low=50.0, high=200.0):
    """Map [0,1] → [50,200]."""
    return X * (high - low) + low

# Image arrays in [0,1]
mnist_Xtr01 = normalize_01(mnist_Xtr)
mnist_Xva01 = normalize_01(mnist_Xva)
mnist_Xte01 = normalize_01(mnist_Xte)

cifar_Xtr01 = normalize_01(cifar_Xtr)
cifar_Xva01 = normalize_01(cifar_Xva)
cifar_Xte01 = normalize_01(cifar_Xte)

# Flat versions for dense AEs
mnist_Xtr01f = mnist_Xtr01.reshape(-1, IMG_DIM)
mnist_Xva01f = mnist_Xva01.reshape(-1, IMG_DIM)
mnist_Xte01f = mnist_Xte01.reshape(-1, IMG_DIM)

cifar_Xtr01f = cifar_Xtr01.reshape(-1, IMG_DIM)
cifar_Xva01f = cifar_Xva01.reshape(-1, IMG_DIM)
cifar_Xte01f = cifar_Xte01.reshape(-1, IMG_DIM)

# 4-D versions for CNN AE
mnist_Xtr4d = mnist_Xtr01[..., np.newaxis]   # (N,28,28,1)
mnist_Xva4d = mnist_Xva01[..., np.newaxis]
mnist_Xte4d = mnist_Xte01[..., np.newaxis]

cifar_Xtr4d = cifar_Xtr01[..., np.newaxis]
cifar_Xva4d = cifar_Xva01[..., np.newaxis]
cifar_Xte4d = cifar_Xte01[..., np.newaxis]

print('Task 3 data shapes ready.')
print(f'  MNIST CNN input: {mnist_Xtr4d.shape}')
print(f'  CIFAR CNN input: {cifar_Xtr4d.shape}')

In [ ]:
# ─── Model A: Deep Convolutional Autoencoder (30-D latent) ────────────────────

def build_cnn_ae(latent_dim=30, img_shape=(28, 28, 1)):
    """
    Encoder: Conv2D → MaxPool (×2) → Flatten → Dense(latent_dim)
    Decoder: Dense → Reshape → UpSampling + Conv2DTranspose (×2)
    """
    inp = layers.Input(shape=img_shape)

    # ── Encoder ──────────────────────────────────────────────────────────────
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inp)   # 28×28×32
    x = layers.MaxPooling2D(2, padding='same')(x)                       # 14×14×32
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)     # 14×14×64
    x = layers.MaxPooling2D(2, padding='same')(x)                       #  7×7×64
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)     #  7×7×64
    x = layers.Flatten()(x)                                             # 7×7×64 = 3136
    encoded = layers.Dense(latent_dim, name='latent')(x)                # 30

    # ── Decoder ──────────────────────────────────────────────────────────────
    x = layers.Dense(7 * 7 * 64, activation='relu')(encoded)
    x = layers.Reshape((7, 7, 64))(x)
    x = layers.Conv2DTranspose(64, 3, activation='relu', padding='same')(x)  # 7×7×64
    x = layers.UpSampling2D(2)(x)                                             # 14×14×64
    x = layers.Conv2DTranspose(32, 3, activation='relu', padding='same')(x)  # 14×14×32
    x = layers.UpSampling2D(2)(x)                                             # 28×28×32
    decoded = layers.Conv2DTranspose(1, 3, activation='sigmoid',
                                      padding='same', name='output')(x)       # 28×28×1

    model = Model(inp, decoded, name='DeepCNN_AE')
    model.compile(optimizer='adam', loss='mse')
    return model

mnist_cnn_ae  = build_cnn_ae()
cifar_cnn_ae  = build_cnn_ae()
mnist_cnn_ae.summary()

In [ ]:
# ─── Model B: Single hidden layer AE (sigmoid encoder, linear decoder) ────────

def build_single_ae(input_dim=IMG_DIM, latent_dim=30):
    """
    input → Dense(30, sigmoid) → Dense(784, linear)
    """
    inp     = layers.Input(shape=(input_dim,))
    encoded = layers.Dense(latent_dim, activation='sigmoid', name='latent')(inp)
    decoded = layers.Dense(input_dim, activation='linear', name='output')(encoded)
    model = Model(inp, decoded, name='Single_AE')
    model.compile(optimizer='adam', loss='mse')
    return model


# ─── Model C: 3-layer AE, 10 nodes per layer, sigmoid encoder, linear decoder ─

def build_3layer_ae(input_dim=IMG_DIM, nodes_per_layer=10):
    """
    Symmetric 3-hidden-layer autoencoder.
    Encoder: Dense(10,σ) → Dense(10,σ) → Dense(10,σ)  [bottleneck]
    Decoder: Dense(10,σ) → Dense(10,σ) → Dense(784, linear)
    Total encoder hidden nodes = 30, distributed equally (10 × 3).
    """
    inp = layers.Input(shape=(input_dim,))
    # Encoder
    x = layers.Dense(nodes_per_layer, activation='sigmoid')(inp)
    x = layers.Dense(nodes_per_layer, activation='sigmoid')(x)
    z = layers.Dense(nodes_per_layer, activation='sigmoid', name='latent')(x)
    # Decoder
    x = layers.Dense(nodes_per_layer, activation='sigmoid')(z)
    x = layers.Dense(nodes_per_layer, activation='sigmoid')(x)
    out = layers.Dense(input_dim, activation='linear', name='output')(x)
    model = Model(inp, out, name='ThreeLayer_AE')
    model.compile(optimizer='adam', loss='mse')
    return model

print('Model architectures defined.')

In [ ]:
# ─── Generic training function ────────────────────────────────────────────────

def train_ae(model, Xtr, Xva, epochs=80, batch_size=256, name=''):
    cb = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    history = model.fit(
        Xtr, Xtr,
        validation_data=(Xva, Xva),
        epochs=epochs, batch_size=batch_size,
        callbacks=[cb], verbose=0
    )
    best_val = min(history.history['val_loss'])
    print(f'  [{name}] Trained {len(history.history["val_loss"])} epochs | '
          f'best val MSE: {best_val:.5f}')
    return history


def eval_ae_snr(model, Xte, is_cnn=False, low=50.0, high=200.0):
    """
    Reconstruct test images and compute average SNR (dB)
    w.r.t. the original images in [50,200] range.
    """
    recon = model.predict(Xte, verbose=0)
    # Undo [0,1] normalisation
    orig_scaled  = denormalize_01(Xte,  low, high)
    recon_scaled = denormalize_01(recon, low, high)
    return compute_snr_db(orig_scaled, recon_scaled)


# ─── Train all models for MNIST ──────────────────────────────────────────────
print('Training MNIST autoencoders (Task 3)...')
train_ae(mnist_cnn_ae,  mnist_Xtr4d,  mnist_Xva4d,  name='MNIST CNN AE')

mnist_single_ae = build_single_ae()
train_ae(mnist_single_ae, mnist_Xtr01f, mnist_Xva01f, name='MNIST Single AE')

mnist_3l_ae = build_3layer_ae()
train_ae(mnist_3l_ae, mnist_Xtr01f, mnist_Xva01f, name='MNIST 3-Layer AE')

# ─── Train all models for CIFAR-10 ───────────────────────────────────────────
print('\nTraining CIFAR-10 autoencoders (Task 3)...')
train_ae(cifar_cnn_ae,  cifar_Xtr4d,  cifar_Xva4d,  name='CIFAR CNN AE')

cifar_single_ae = build_single_ae()
train_ae(cifar_single_ae, cifar_Xtr01f, cifar_Xva01f, name='CIFAR Single AE')

cifar_3l_ae = build_3layer_ae()
train_ae(cifar_3l_ae, cifar_Xtr01f, cifar_Xva01f, name='CIFAR 3-Layer AE')

print('\nAll Task 3 models trained.')

In [ ]:
# ─── Task 3: Evaluate SNR for all models ─────────────────────────────────────
results_task3 = {}

for dataset, cnn_ae, single_ae, three_ae, Xte4d, Xte01f in [
    ('MNIST',    mnist_cnn_ae,  mnist_single_ae,  mnist_3l_ae,
                 mnist_Xte4d,   mnist_Xte01f),
    ('CIFAR-10', cifar_cnn_ae,  cifar_single_ae,  cifar_3l_ae,
                 cifar_Xte4d,   cifar_Xte01f)
]:
    snr_cnn    = eval_ae_snr(cnn_ae,    Xte4d)
    snr_single = eval_ae_snr(single_ae, Xte01f)
    snr_3l     = eval_ae_snr(three_ae,  Xte01f)
    results_task3[dataset] = dict(cnn=snr_cnn, single=snr_single, three=snr_3l)
    print(f'[{dataset}]')
    print(f'  Deep CNN AE              : {snr_cnn:.2f} dB')
    print(f'  Single hidden layer AE   : {snr_single:.2f} dB')
    print(f'  3-layer AE (10+10+10)    : {snr_3l:.2f} dB')

print('\n' + '=' * 65)
print(f'{"Dataset":<12} {"Deep CNN AE":>15} {"Single AE":>12} {"3-Layer AE":>12}')
print('-' * 65)
for ds, vals in results_task3.items():
    print(f'{ds:<12} {vals["cnn"]:>14.2f} dB {vals["single"]:>11.2f} dB '
          f'{vals["three"]:>11.2f} dB')
print('=' * 65)

In [ ]:
# ─── Task 3: Visualise reconstructions ───────────────────────────────────────

def show_task3_reconstructions(models_dict, Xte4d, Xte01f, n_imgs=8,
                                dataset_name='Dataset'):
    """
    Display original + 3 reconstructions side by side.
    models_dict: {'CNN AE': (model, input_data), ...}
    """
    n_rows = len(models_dict) + 1
    fig, axes = plt.subplots(n_rows, n_imgs, figsize=(n_imgs * 1.5, n_rows * 1.8))
    fig.suptitle(f'{dataset_name} — Reconstruction Comparison (Task 3)', fontsize=12)

    # Row 0: Originals
    orig = denormalize_01(Xte01f[:n_imgs]).reshape(n_imgs, 28, 28)
    for i in range(n_imgs):
        axes[0, i].imshow(orig[i], cmap='gray', vmin=50, vmax=200)
        axes[0, i].axis('off')
    axes[0, 0].set_ylabel('Original', fontsize=8)

    for row_idx, (label, (model, X_in)) in enumerate(models_dict.items(), start=1):
        recon = model.predict(X_in[:n_imgs], verbose=0)
        recon_img = denormalize_01(recon).reshape(n_imgs, 28, 28)
        for i in range(n_imgs):
            axes[row_idx, i].imshow(recon_img[i], cmap='gray', vmin=50, vmax=200)
            axes[row_idx, i].axis('off')
        axes[row_idx, 0].set_ylabel(label, fontsize=7)

    plt.tight_layout()
    fname = dataset_name.lower().replace('-', '_').replace(' ', '_')
    plt.savefig(f'{fname}_task3_reconstructions.png', dpi=150, bbox_inches='tight')
    plt.show()


show_task3_reconstructions(
    {'Deep CNN AE':           (mnist_cnn_ae,    mnist_Xte4d),
     'Single AE (sig+lin)':   (mnist_single_ae, mnist_Xte01f),
     '3-Layer AE (10+10+10)': (mnist_3l_ae,     mnist_Xte01f)},
    mnist_Xte4d, mnist_Xte01f, dataset_name='MNIST'
)

show_task3_reconstructions(
    {'Deep CNN AE':           (cifar_cnn_ae,    cifar_Xte4d),
     'Single AE (sig+lin)':   (cifar_single_ae, cifar_Xte01f),
     '3-Layer AE (10+10+10)': (cifar_3l_ae,     cifar_Xte01f)},
    cifar_Xte4d, cifar_Xte01f, dataset_name='CIFAR-10'
)

In [ ]:
# ─── Task 3: SNR bar chart ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Task 3 — Average Test SNR Comparison (dB)', fontsize=13)
models_labels = ['Deep CNN AE', 'Single AE\n(sig+lin, 30)', '3-Layer AE\n(10+10+10)']
colors = ['#2196F3', '#FF9800', '#4CAF50']

for ax, (ds, vals) in zip(axes, results_task3.items()):
    snrs = [vals['cnn'], vals['single'], vals['three']]
    bars = ax.bar(models_labels, snrs, color=colors, edgecolor='black', width=0.5)
    ax.set_title(ds, fontsize=12)
    ax.set_ylabel('SNR (dB)')
    ax.set_ylim(0, max(snrs) * 1.2)
    for bar, v in zip(bars, snrs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                f'{v:.2f} dB', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('task3_snr_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Final Summary Table

Comprehensive comparison across all tasks.

In [ ]:
# ─── Grand summary ────────────────────────────────────────────────────────────
print('=' * 90)
print('FINAL SUMMARY — All Tasks')
print('=' * 90)
print(f'{"Dataset":<12} {"Task":<6} {"Model / Method":<38} {"Test Acc":>10} {"SNR (dB)":>10}')
print('-' * 90)

rows_final = [
    # MNIST
    ('MNIST', 'T1', 'Standard PCA (30 comps) + LR',    f"{mnist_pca_res['acc']*100:.2f}%",   f"{mnist_pca_res['snr']:.2f}"),
    ('MNIST', 'T1', 'Randomized PCA (30 comps) + LR',  f"{mnist_rpca_res['acc']*100:.2f}%",  f"{mnist_rpca_res['snr']:.2f}"),
    ('MNIST', 'T2', 'Tied Linear AE (30, linear) + LR',f"{mnist_ae2_acc*100:.2f}%",          f"{mnist_ae2_snr:.2f}"),
    ('MNIST', 'T3', 'Deep CNN AE (30-D latent)',         '—',                                  f"{results_task3['MNIST']['cnn']:.2f}"),
    ('MNIST', 'T3', 'Single AE (sigmoid+linear, 30)',   '—',                                  f"{results_task3['MNIST']['single']:.2f}"),
    ('MNIST', 'T3', '3-Layer AE (10+10+10, sig+lin)',   '—',                                  f"{results_task3['MNIST']['three']:.2f}"),
    # CIFAR-10
    ('CIFAR-10','T1','Standard PCA (30 comps) + LR',    f"{cifar_pca_res['acc']*100:.2f}%",   f"{cifar_pca_res['snr']:.2f}"),
    ('CIFAR-10','T1','Randomized PCA (30 comps) + LR',  f"{cifar_rpca_res['acc']*100:.2f}%",  f"{cifar_rpca_res['snr']:.2f}"),
    ('CIFAR-10','T2','Tied Linear AE (30, linear) + LR',f"{cifar_ae2_acc*100:.2f}%",          f"{cifar_ae2_snr:.2f}"),
    ('CIFAR-10','T3','Deep CNN AE (30-D latent)',         '—',                                  f"{results_task3['CIFAR-10']['cnn']:.2f}"),
    ('CIFAR-10','T3','Single AE (sigmoid+linear, 30)',   '—',                                  f"{results_task3['CIFAR-10']['single']:.2f}"),
    ('CIFAR-10','T3','3-Layer AE (10+10+10, sig+lin)',   '—',                                  f"{results_task3['CIFAR-10']['three']:.2f}"),
]

for ds, task, method, acc, snr in rows_final:
    print(f'{ds:<12} {task:<6} {method:<38} {acc:>10} {snr:>10}')

print('=' * 90)

---
## Discussion and Conclusions

### Task 1 — PCA vs Randomized PCA
- **Classification:** Both methods produce almost identical classification accuracy because Randomized PCA (TruncatedSVD) approximates the same top-$K$ eigenvectors with $O(nd K)$ instead of $O(d^3)$ complexity.
- **SNR:** With only 30 principal components out of 784, a significant portion of the signal energy is discarded. MNIST has more compact structure → higher SNR than CIFAR-10 which has more complex textures.
- **ROC Curves:** MNIST macro-AUC is significantly higher than CIFAR-10, reflecting MNIST's structural regularity.

### Task 2 — Tied Linear Autoencoder
- **Theoretical equivalence with PCA:** A single-layer tied linear AE with MSE loss converges to a solution whose weight columns span the **same subspace** as the top-30 PCA eigenvectors. This is proven by Baldi & Hornik (1989).
- **Why not identical?** The solution is unique only up to an orthogonal rotation within the 30-D subspace. SGD finds a specific rotation that may differ from the PCA solution.
- **Classification:** Performance is comparable to PCA since both extract essentially the same features (same subspace).

### Task 3 — SNR Comparison
| Architecture | Expected SNR | Reason |
|---|---|---|
| **Deep CNN AE** | Highest | Convolutional layers exploit spatial structure, much more expressive. |
| **Single AE (sig, 30)** | Middle | Dense connection with non-linearity, more capacity than linear PCA. |
| **3-Layer AE (10+10+10)** | Lowest among three | Bottleneck is only 10-D; information loss is severe. |

> **Key insight:** Distributing 30 nodes across 3 layers creates a **more constrained bottleneck** (10-D) compared to a single 30-node layer. Even though the total parameter count may be similar, the information bottleneck is narrower, leading to lower SNR. The deep CNN AE wins because convolutions are fundamentally better suited to image data — they exploit local spatial correlations and translation equivariance.